In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
df=pd.read_excel("Fatigue.xlsx")
df.to_csv("Fatigue.csv", index=False)

FileNotFoundError: [Errno 2] No such file or directory: 'Fatigue.xlsx'

In [ ]:


# Rename columns (adjust based on your exact names)
df = df.rename(columns={
    "Body weight (kg)": "body_weight",
    "What is your squat 1RM?": "squat_1rm",
    "What is your bench press 1RM?": "bench_1rm",
    "What is your deadlift 1RM?": "deadlift_1rm",
    "How many days per week do you train?": "frequency_per_week",
    "How long have you been training? (months)": "experience_months",
    "Have you taken a break (months)?": "detraining_months",
    "What was your maximum body weight?": "max_body_weight",
    "Rate your technique (1-5)": "technique_score"
})

df.head()

,Timestamp,Name\n,AGE,Body Weight,How long have you been doing strength training?,Training Frequency(weekly),Have you taken a break (detraining)?,Rate your form (Bench / Squat / Deadlift),Your all time 1 Rep Max(Bench Press) in Kgs,Your current 1 Rep Max(Bench Press) in Kgs,Your all time 1 Rep Max(Deadlift) in Kgs,Your current 1 Rep Max(Deadlift) in Kgs,Your all time 1 Rep Max(Squats) in Kgs,Your current 1 Rep Max(Squats) in Kgs,How confident are you with your 1RM values?
0,2026-04-09 05:44:33.257,Raunak Suman,21,78,1-3 years,4-5 days,Yes (1-3 months),4,75,65,160,130,120,110,3
1,2026-04-09 05:50:45.588,Aayush soni,22,74,1-3 years,4-5 days,Yes (>3 months),4,95kg,80kg,160kg,120-130kg,125kg,90-95kg,4
2,2026-04-09 06:22:02.948,PONUGOTI DHEERAJ,19,67,6-12 months,4-5 days,Yes (<1month),4,100kgs,85 kgs,160 kgs,150 kgs,100 kgs,90 kgs,4
3,2026-04-09 06:23:25.381,Siddharth K,19,63,3+ years,6+ days,No,4,65,60,140,140,120,100,4
4,2026-04-09 06:26:10.753,D Mahadev,19,71,6-12 months,4-5 days,Yes (>3 months),4,80,60,130,120,100,90,4


In [ ]:
df.columns = [
    "timestamp",
    "name",
    "age",
    "body_weight",
    "experience_text",
    "frequency_text",
    "detraining_text",
    "technique_score",
    "bench_1rm_all",
    "bench_1rm_current",
    "deadlift_1rm_all",
    "deadlift_1rm_current",
    "squat_1rm_all",
    "squat_1rm_current",
    "confidence_1rm"
]

df

,timestamp,name,age,body_weight,experience_text,frequency_text,detraining_text,technique_score,bench_1rm_all,bench_1rm_current,deadlift_1rm_all,deadlift_1rm_current,squat_1rm_all,squat_1rm_current,confidence_1rm
0,2026-04-09 05:44:33.257,Raunak Suman,21,78,1-3 years,4-5 days,Yes (1-3 months),4,75,65,160,130,120,110,3
1,2026-04-09 05:50:45.588,Aayush soni,22,74,1-3 years,4-5 days,Yes (>3 months),4,95kg,80kg,160kg,120-130kg,125kg,90-95kg,4
2,2026-04-09 06:22:02.948,PONUGOTI DHEERAJ,19,67,6-12 months,4-5 days,Yes (<1month),4,100kgs,85 kgs,160 kgs,150 kgs,100 kgs,90 kgs,4
3,2026-04-09 06:23:25.381,Siddharth K,19,63,3+ years,6+ days,No,4,65,60,140,140,120,100,4
4,2026-04-09 06:26:10.753,D Mahadev,19,71,6-12 months,4-5 days,Yes (>3 months),4,80,60,130,120,100,90,4
5,2026-04-09 07:04:14.491,Rajneesh Raghuwanshi,22,69,1-3 years,4-5 days,Yes (>3 months),4,85,80,160,NaN,100,NaN,5
6,2026-04-09 07:06:21.898,K Sricharan,18,73-74,1-3 years,4-5 days,Yes (<1month),4,85,85,155,155,120,IDK,5
7,2026-04-09 07:22:09.547,Hrishikesh Kurapati,20,74.8,1-3 years,4-5 days,No,4,90,85,170,170,140,140,5
8,2026-04-09 08:19:12.989,Kartik Raghuwanshi,21,67,1-3 years,4-5 days,No,5,85,70,140,120,110,80,5
9,2026-04-09 08:49:07.896,Praveen,21,75,1-3 years,6+ days,No,5,100kg,100kg,180kg,180kg,150kg,150kg,3


In [ ]:
df.loc[6, "body_weight"] = 73.5

In [ ]:
def clean_weight(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).lower().strip()
    
    # remove units
    x = x.replace("kgs", "").replace("kg", "").strip()
    
    # handle ranges like 120-130
    if "-" in x:
        parts = x.split("-")
        nums = []
        for p in parts:
            p = p.strip()
            if p != "":
                try:
                    nums.append(float(p))
                except:
                    pass
        return np.mean(nums) if len(nums) > 0 else np.nan
    
    # single value
    try:
        return float(x)
    except:
        return np.nan

In [ ]:
cols = [
    "bench_1rm_all", "bench_1rm_current",
    "deadlift_1rm_all", "deadlift_1rm_current",
    "squat_1rm_all", "squat_1rm_current"
]

for col in cols:
    df[col] = df[col].apply(clean_weight)

In [ ]:
df[cols]

,bench_1rm_all,bench_1rm_current,deadlift_1rm_all,deadlift_1rm_current,squat_1rm_all,squat_1rm_current
0,75.0,65.0,160.0,130.0,120.0,110.0
1,95.0,80.0,160.0,125.0,125.0,92.5
2,100.0,85.0,160.0,150.0,100.0,90.0
3,65.0,60.0,140.0,140.0,120.0,100.0
4,80.0,60.0,130.0,120.0,100.0,90.0
5,85.0,80.0,160.0,NaN,100.0,NaN
6,85.0,85.0,155.0,155.0,120.0,NaN
7,90.0,85.0,170.0,170.0,140.0,140.0
8,85.0,70.0,140.0,120.0,110.0,80.0
9,100.0,100.0,180.0,180.0,150.0,150.0


In [ ]:
# Fill missing current values as 90% of all-time max
df["bench_1rm_current"] = df["bench_1rm_current"].fillna(0.9 * df["bench_1rm_all"])
df["deadlift_1rm_current"] = df["deadlift_1rm_current"].fillna(0.9 * df["deadlift_1rm_all"])
df["squat_1rm_current"] = df["squat_1rm_current"].fillna(0.9 * df["squat_1rm_all"])

In [ ]:
df[[
    "bench_1rm_all","bench_1rm_current",
    "deadlift_1rm_all","deadlift_1rm_current",
    "squat_1rm_all","squat_1rm_current"
]]

,bench_1rm_all,bench_1rm_current,deadlift_1rm_all,deadlift_1rm_current,squat_1rm_all,squat_1rm_current
0,75.0,65.0,160.0,130.0,120.0,110.0
1,95.0,80.0,160.0,125.0,125.0,92.5
2,100.0,85.0,160.0,150.0,100.0,90.0
3,65.0,60.0,140.0,140.0,120.0,100.0
4,80.0,60.0,130.0,120.0,100.0,90.0
5,85.0,80.0,160.0,144.0,100.0,90.0
6,85.0,85.0,155.0,155.0,120.0,108.0
7,90.0,85.0,170.0,170.0,140.0,140.0
8,85.0,70.0,140.0,120.0,110.0,80.0
9,100.0,100.0,180.0,180.0,150.0,150.0


In [ ]:
df["body_weight"] = pd.to_numeric(df["body_weight"], errors="coerce")

In [ ]:
# strength
df["bench_rel"] = df["bench_1rm_current"] / df["body_weight"]
df["squat_rel"] = df["squat_1rm_current"] / df["body_weight"]
df["deadlift_rel"] = df["deadlift_1rm_current"] / df["body_weight"]

df["strength_score"] = df[["bench_rel","squat_rel","deadlift_rel"]].mean(axis=1)

In [ ]:
df[["body_weight","strength_score"]]

,body_weight,strength_score
0,78.0,1.303419
1,74.0,1.340090
2,67.0,1.616915
3,63.0,1.587302
4,71.0,1.267606
5,69.0,1.516908
6,73.5,1.578231
7,74.8,1.760250
8,67.0,1.343284
9,75.0,1.911111


In [ ]:
# Fix swapped values automatically
def fix_swap(all_val, current_val):
    if pd.isna(all_val) or pd.isna(current_val):
        return all_val, current_val
    
    if current_val > all_val:
        return current_val, all_val  # swap
    
    return all_val, current_val


for lift in ["bench", "squat", "deadlift"]:
    df[[f"{lift}_1rm_all", f"{lift}_1rm_current"]] = df.apply(
        lambda row: pd.Series(
            fix_swap(row[f"{lift}_1rm_all"], row[f"{lift}_1rm_current"])
        ),
        axis=1
    )

In [ ]:
df["bench_loss"] = (df["bench_1rm_all"] - df["bench_1rm_current"]) / df["bench_1rm_all"]
df["squat_loss"] = (df["squat_1rm_all"] - df["squat_1rm_current"]) / df["squat_1rm_all"]
df["deadlift_loss"] = (df["deadlift_1rm_all"] - df["deadlift_1rm_current"]) / df["deadlift_1rm_all"]

df["strength_loss"] = df[["bench_loss", "squat_loss", "deadlift_loss"]].mean(axis=1)

df[["strength_loss"]]

,strength_loss
0,0.134722
1,0.212215
2,0.104167
3,0.081197
4,0.142308
5,0.086275
6,0.033333
7,0.018519
8,0.197352
9,0.000000


In [ ]:
# Experience → months
exp_map = {
    "0-3 months": 3,
    "3-6 months": 6,
    "6-12 months": 12,
    "1-3 years": 24,
    "3+ years": 48
}

df["experience_months"] = df["experience_text"].map(exp_map)

In [ ]:
freq_map = {
    "0-1 days": 1,
    "2-3 days": 3,
    "4-5 days": 5,
    "6+ days": 6
}

df["frequency_per_week"] = df["frequency_text"].map(freq_map)

In [ ]:
det_map = {
    "No": 0,
    "Yes (<1month)": 0.5,
    "Yes (1-3 months)": 2,
    "Yes (>3 months)": 4
}

df["detraining_months"] = df["detraining_text"].map(det_map)

In [ ]:
df[["experience_months", "frequency_per_week", "detraining_months"]]

,experience_months,frequency_per_week,detraining_months
0,24,5,2.0
1,24,5,4.0
2,12,5,0.5
3,48,6,0.0
4,12,5,4.0
5,24,5,4.0
6,24,5,0.5
7,24,5,0.0
8,24,5,0.0
9,24,6,0.0


In [ ]:
features = [
    "strength_score",
    "strength_loss",
    "experience_months",
    "frequency_per_week",
    "technique_score",
    "confidence_1rm",
    "detraining_months"
]

df[features]

,strength_score,strength_loss,experience_months,frequency_per_week,technique_score,confidence_1rm,detraining_months
0,1.303419,0.134722,24,5,4,3,2.0
1,1.340090,0.212215,24,5,4,4,4.0
2,1.616915,0.104167,12,5,4,4,0.5
3,1.587302,0.081197,48,6,4,4,0.0
4,1.267606,0.142308,12,5,4,4,4.0
5,1.516908,0.086275,24,5,4,5,4.0
6,1.578231,0.033333,24,5,4,5,0.5
7,1.760250,0.018519,24,5,4,5,0.0
8,1.343284,0.197352,24,5,5,5,0.0
9,1.911111,0.000000,24,6,5,3,0.0


In [ ]:
from sklearn.preprocessing import MinMaxScaler

features = [
    "strength_score",
    "strength_loss",
    "experience_months",
    "frequency_per_week",
    "technique_score",
    "confidence_1rm",
    "detraining_months"
]

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(df[features])

X_scaled = pd.DataFrame(X_scaled, columns=features)

X_scaled

,strength_score,strength_loss,experience_months,frequency_per_week,technique_score,confidence_1rm,detraining_months
0,0.055653,0.634839,0.333333,0.0,0.0,0.0,0.500
1,0.112640,1.000000,0.333333,0.0,0.0,0.5,1.000
2,0.542823,0.490855,0.000000,0.0,0.0,0.5,0.125
3,0.496804,0.382615,1.000000,1.0,0.0,0.5,0.000
4,0.000000,0.670583,0.000000,0.0,0.0,0.5,1.000
5,0.387413,0.406543,0.333333,0.0,0.0,1.0,1.000
6,0.482709,0.157073,0.333333,0.0,0.0,1.0,0.125
7,0.765563,0.087263,0.333333,0.0,0.0,1.0,0.000
8,0.117603,0.929961,0.333333,0.0,1.0,1.0,0.000
9,1.000000,0.000000,0.333333,1.0,1.0,0.0,0.000


In [ ]:
weights = {
    "strength_score": 0.35,
    "strength_loss": -0.20,
    "experience_months": 0.15,
    "frequency_per_week": 0.10,
    "technique_score": 0.10,
    "confidence_1rm": 0.05,
    "detraining_months": -0.05
}

df["final_score"] = sum(X_scaled[col] * w for col, w in weights.items())

df[["final_score"]]

,final_score
0,-0.082489
1,-0.135576
2,0.110567
3,0.372358
4,-0.159117
5,0.104286
6,0.231283
7,0.350494
8,0.055169
9,0.600000


In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

df[["final_score", "cluster"]]

,final_score,cluster
0,-0.082489,1
1,-0.135576,1
2,0.110567,0
3,0.372358,2
4,-0.159117,1
5,0.104286,1
6,0.231283,0
7,0.350494,0
8,0.055169,0
9,0.600000,2


In [ ]:
df.groupby("cluster")[[
    "strength_score",
    "strength_loss",
    "experience_months",
    "frequency_per_week",
    "technique_score",
    "confidence_1rm",
    "detraining_months",
    "final_score"
]].mean()


,strength_score,strength_loss,experience_months,frequency_per_week,technique_score,confidence_1rm,detraining_months,final_score
cluster,,,,,,,,
0,1.611849,0.077735,21.6,5.0,4.200000,4.6,0.200000,0.211472
1,1.357006,0.143880,21.0,5.0,4.000000,4.0,3.500000,-0.068224
2,1.685263,0.027066,32.0,6.0,4.333333,4.0,0.666667,0.434988


In [ ]:
df_sorted = df.sort_values(by="final_score", ascending=False)

df_sorted[["name", "final_score", "cluster"]]

,name,final_score,cluster
9,Praveen,0.600000,2
3,Siddharth K,0.372358,2
7,Hrishikesh Kurapati,0.350494,0
11,Aadesh Ashok Phadake,0.332605,2
10,MUDE DINESH NAIK,0.309846,0
6,K Sricharan,0.231283,0
2,PONUGOTI DHEERAJ,0.110567,0
5,Rajneesh Raghuwanshi,0.104286,1
8,Kartik Raghuwanshi,0.055169,0
0,Raunak Suman,-0.082489,1
